<div style="background:linear-gradient(135deg,#0d1117,#13243b,#0f3a5f);padding:44px 38px;border-radius:16px;color:#f0f6fc;font-family:'Segoe UI',sans-serif;border:1px solid #30363d;">
  <div style="font-size:.8em;letter-spacing:3px;opacity:.65;text-transform:uppercase;margin-bottom:10px;">Causal Attribution · Region Occlusion · Inference Only</div>
  <h1 style="font-size:2.0em;margin:0 0 12px 0;font-weight:700;line-height:1.25;color:#f0f6fc !important;">Model Gerçekte Neye Bakıyor?</h1>
  <h2 style="font-size:1.05em;font-weight:300;opacity:.82;margin:0 0 22px 0;line-height:1.5;color:#f0f6fc !important;">Bölge kapatma ile nedensel bağımlılık ölçümü — alan eşleşmeli rastgele kontrollerle</h2>
  <hr style="border:0;border-top:1px solid #30363d;margin:0 0 18px 0;">
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;font-size:.88em;opacity:.88;">
    <div><b>Eğitim:</b> yok — kayıtlı kontrol noktaları</div>
    <div><b>Süre:</b> ~15 dakika</div>
    <div><b>Kümeler:</b> iç test + RSNA + NIH</div>
    <div><b>Koşul:</b> 11 oklüzyon × 3 kol</div>
  </div>
  <div style="margin-top:20px;padding:13px 16px;background:rgba(56,139,253,.10);border-left:4px solid #388bfd;border-radius:4px;font-size:.86em;line-height:1.55;">
    <b>Gerekçe:</b> Dikkat analizi, segmentasyonsuz modelin akciğerle örtüşmesinin şans düzeyinin 22 katı altında olduğunu gösterdi. Ancak dikkat haritaları <b>korelasyoneldir</b>: modelin bir bölgeye bakması, kararını oraya dayandırdığını kanıtlamaz. Bu notebook nedensel testi yapar — bir bölge kapatılır ve başarım düşüşü ölçülür. Kritik nokta: <b>her oklüzyona aynı alanı kaplayan rastgele bir kontrol eşlik eder</b>, çünkü herhangi bir bölgeyi kapatmak girdi dağılımını bozarak zaten bir miktar düşüş yaratır. Anlamlı olan, hedefli oklüzyonun alan eşleşmeli rastgele oklüzyondan <i>ne kadar fazla</i> zarar verdiğidir.
  </div>
</div>

## Yöntem

Her görüntü, kolun kendi ön-işlemesinden geçtikten sonra $224\times224$ uzayında
kapatılır; kapatılan bölge veri kümesi ortalama grisiyle ($\approx 122$) doldurulur —
maskeleme hattındaki dolgu değeriyle aynıdır.

### Koşullar

| Koşul | Bölge | Yanıtladığı soru |
|---|---|---|
| `baseline` | — | Referans başarım |
| **`lungs`** | Akciğer maskesi | **Model akciğere gerçekten bağımlı mı?** |
| `non_lungs` | Maskenin tümleyeni | Karar akciğer dışında mı? |
| `corners` | Dört köşe karesi | Köşe artefaktı bağımlılığı |
| `border` | Çevre şeridi | Kenar/çerçeve bağımlılığı |
| `subdiaphragm` | Alt bant (diyafram altı) | Toplu dikkatin yoğunlaştığı bölge |
| `upper` | Üst bant (omuz/klavikula) | Kontrol bölgesi |
| `center` | Merkezî dikey sütun | Omurga/mediasten bağımlılığı |
| `rand_lungs` | **Akciğerle aynı alanda rastgele dikdörtgen** | `lungs` için alan eşleşmeli kontrol |
| `rand_20` / `rand_30` | %20 / %30 rastgele dikdörtgen | Diğer koşullar için kontrol |

### Okuma kuralı

$$\Delta\mathrm{AUC}_{\text{bölge}} = \mathrm{AUC}_{\text{baseline}} - \mathrm{AUC}_{\text{bölge kapalı}}$$

Bir bölgenin **nedensel katkısı**, kendi düşüşü ile alan eşleşmeli rastgele kontrolün
düşüşü arasındaki farktır:

$$\text{net etki} = \Delta\mathrm{AUC}_{\text{bölge}} - \Delta\mathrm{AUC}_{\text{rastgele, aynı alan}}$$

Net etki sıfıra yakınsa, o bölge kararda rastgele bir bölgeden daha fazla rol oynamıyor
demektir. **Beklenen bulgu:** A kolunda `lungs` oklüzyonunun net etkisi ≈ 0
(akciğeri silmek zarar vermiyor), `corners` / `border` / `subdiaphragm` net etkisi > 0.
C kolunda tam tersi.

### Beklenen tutarlılık denetimleri

- C kolunda `non_lungs` oklüzyonu **etkisiz** olmalıdır (o bölge zaten gri doludur).
- C kolunda `lungs` oklüzyonu **yıkıcı** olmalıdır (tüm bilgi silinir).

Bu ikisi tutmuyorsa hatta bir sorun vardır.

## Kurulum

| # | Sekme | Kimlik |
|---|---|---|
| 1 | Datasets | `yusufmurtaza01/chest-xray-pneumonia-balanced-dataset` |
| 2 | **Notebooks** | `segmentation-ablation-lung-focused-vit` |
| 3 | Competitions | `rsna-pneumonia-detection-challenge` |
| 4 | Datasets | `nih-chest-xrays/data` |

GPU + Internet açık.

In [ ]:
import os, re, io, gc, json, glob, time, random, zipfile, hashlib, warnings, types
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from collections import OrderedDict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from sklearn.metrics import roc_curve, auc
from scipy import stats as sp_stats

# ----------------------------- AYARLAR -----------------------------
SEED            = 42
MAX_PER_CLASS   = 250      # dis setlerden sinif basina
N_BOOT          = 1500
ARMS            = ["raw", "roi", "lung"]
# --------------------------------------------------------------------

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ARM_LABEL = OrderedDict([("raw", "A · Segmentasyonsuz"),
                         ("roi", "B · Yalnizca RoI kirpma"),
                         ("lung", "C · Maske + RoI (onerilen)")])
ARM_COLOR = {"raw": "#B04A1E", "roi": "#C2900A", "lung": "#0D8FA2"}

plt.rcParams.update({
    "figure.dpi": 130, "figure.facecolor": "white", "savefig.facecolor": "white",
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#8A9499", "axes.labelcolor": "#1B2327", "text.color": "#1B2327",
    "xtick.color": "#5A686F", "ytick.color": "#5A686F",
    "grid.color": "#D8DFE1", "grid.linewidth": 0.7, "legend.frameon": False,
})
WORK = "/kaggle/working"
print(f"Cihaz: {device} | Torch {torch.__version__}")

In [ ]:
# ── Girdiler ─────────────────────────────────────────────────────────────
INPUT = "/kaggle/input"

def find_classification_root(root=INPUT):
    hits = []
    for r, dirs, _ in os.walk(root):
        if os.path.basename(r) == "train" and {"NORMAL", "PNEUMONIA"} <= set(dirs):
            hits.append(os.path.dirname(r))
    hits.sort(key=lambda p: (0 if "balanced" in p.lower() else 1, len(p)))
    return hits[0] if hits else None

def find_dir_with(fname, root=INPUT):
    for r, _, files in os.walk(root):
        if fname in files:
            return r
    return None

BASE      = find_classification_root()
RSNA_BASE = find_dir_with("stage_2_detailed_class_info.csv")
NIH_BASE  = find_dir_with("Data_Entry_2017.csv")

ckpts = {}
for p in glob.glob(os.path.join(INPUT, "**", "*.pth"), recursive=True):
    b = os.path.basename(p).lower()
    for arm in ARMS:
        if f"_{arm}." in b:
            ckpts[arm] = p
missing = [a for a in ARMS if a not in ckpts]
assert not missing, f"Kontrol noktasi eksik: {missing}"
assert BASE is not None, "Siniflandirma veri kumesi bulunamadi."

print("Bulunanlar")
print(f"  veri kumesi : {BASE}")
print(f"  RSNA        : {RSNA_BASE}")
print(f"  NIH         : {NIH_BASE}")
for a in ARMS:
    print(f"  ckpt {a:<5}  : {os.path.basename(ckpts[a])}")

In [ ]:
# ── Modeller + segmentasyon ──────────────────────────────────────────────
def build_vit_inference(num_classes, dropout):
    m = models.vit_b_16(weights=None)
    in_f = m.heads.head.in_features
    m.heads.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_f, num_classes))
    return m

MODELS, CFG, CLASS_TO_IDX = {}, None, None
for arm in ARMS:
    ck = torch.load(ckpts[arm], map_location=device, weights_only=False)
    CFG, CLASS_TO_IDX = ck["config"], ck["class_to_idx"]
    m = build_vit_inference(CFG["num_classes"], CFG.get("dropout", 0.1)).to(device)
    m.load_state_dict(ck["model_state_dict"], strict=True)
    MODELS[arm] = m.eval()
PNEU_IDX = CLASS_TO_IDX["PNEUMONIA"]
S = CFG["img_size"]
FILL = int(round(CFG["mean"][0] * 255))
print(f"Modeller yuklendi | girdi {S}x{S} | dolgu degeri {FILL}")

MEAN_T = torch.tensor(CFG["mean"]).view(3, 1, 1)
STD_T  = torch.tensor(CFG["std"]).view(3, 1, 1)

import transformers
from transformers import AutoModel
print("ianpan/chest-x-ray-basic yukleniyor...")
def _load_seg():
    return AutoModel.from_pretrained("ianpan/chest-x-ray-basic",
                                     trust_remote_code=True).to(device).eval()
_of = getattr(transformers.modeling_utils.PreTrainedModel, "_finalize_model_loading", None)
try:
    if _of is not None:
        def _sf(model, *a, **k):
            if not hasattr(model, "all_tied_weights_keys"):
                model.all_tied_weights_keys = {}
            return _of(model, *a, **k)
        transformers.modeling_utils.PreTrainedModel._finalize_model_loading = _sf
    seg_model = _load_seg()
except Exception as e:
    if "all_tied_weights_keys" in str(e):
        transformers.modeling_utils.PreTrainedModel.all_tied_weights_keys = {}
        seg_model = _load_seg()
    else:
        raise
finally:
    if _of is not None:
        transformers.modeling_utils.PreTrainedModel._finalize_model_loading = _of
print("Segmentasyon modeli hazir.")

In [ ]:
# ── On-isleme (ablasyonla birebir) — maske de doner ──────────────────────
def load_image_any(path, short_max):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".dcm":
        import pydicom
        dcm = pydicom.dcmread(path)
        arr = dcm.pixel_array.astype(np.float32)
        arr -= arr.min()
        if arr.max() > 0:
            arr /= arr.max()
        if getattr(dcm, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
            arr = 1.0 - arr
        pil = Image.fromarray((arr * 255).astype(np.uint8)).convert("RGB")
    else:
        pil = Image.open(path).convert("RGB")
    W0, H0 = pil.size
    short = min(W0, H0)
    if short > short_max:
        s = short_max / short
        pil = pil.resize((int(round(W0 * s)), int(round(H0 * s))), Image.BILINEAR)
    return np.asarray(pil).astype(np.uint8), np.asarray(pil.convert("L"))

@torch.inference_mode()
def lung_mask_ianpan(gray_u8, out_hw):
    x = seg_model.preprocess(gray_u8)
    x = torch.from_numpy(x).unsqueeze(0).unsqueeze(0).float().to(device)
    logits = seg_model(x)["mask"]
    logits = F.interpolate(logits, size=out_hw, mode="bilinear", align_corners=False)
    pred = logits.argmax(dim=1)[0].cpu().numpy()
    return ((pred == 1) | (pred == 2)).astype(np.uint8)

def prep_arms(rgb_u8, lung_u8, cfg):
    H, W = lung_u8.shape
    short = min(H, W); Sz = cfg["img_size"]
    out = {"raw": (cv2.resize(rgb_u8, (Sz, Sz), interpolation=cv2.INTER_AREA),
                   cv2.resize(lung_u8, (Sz, Sz), interpolation=cv2.INTER_NEAREST).astype(np.uint8))}
    if lung_u8.sum() < 1:
        fb = cv2.resize(rgb_u8, (Sz, Sz), interpolation=cv2.INTER_AREA)
        ones = np.ones((Sz, Sz), np.uint8)
        out["roi"] = (fb, ones); out["lung"] = (fb, ones)
        return out
    dil = max(1, int(round(short * cfg["mask_dilate_frac"])))
    kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * dil + 1, 2 * dil + 1))
    mask_d = cv2.dilate(lung_u8, kern, iterations=1)
    soft = mask_d.astype(np.float32)
    f = cfg["mask_feather"]
    if f and f >= 3:
        if f % 2 == 0:
            f += 1
        soft = cv2.GaussianBlur(soft, (f, f), 0)
    soft = np.clip(soft, 0.0, 1.0)[..., None]
    fill = np.array([m * 255.0 for m in cfg["mean"]], dtype=np.float32)
    masked = (rgb_u8.astype(np.float32) * soft +
              fill[None, None, :] * (1.0 - soft)).clip(0, 255).astype(np.uint8)
    ys, xs = np.where(mask_d > 0)
    y0, y1 = int(ys.min()), int(ys.max()); x0, x1 = int(xs.min()), int(xs.max())
    pad = int(round(short * cfg["roi_pad_frac"]))
    y0 = max(0, y0 - pad); x0 = max(0, x0 - pad)
    y1 = min(H - 1, y1 + pad); x1 = min(W - 1, x1 + pad)
    bh, bw = (y1 - y0 + 1), (x1 - x0 + 1)
    side = min(max(bh, bw), min(H, W))
    cy, cx = (y0 + y1) // 2, (x0 + x1) // 2
    ty0 = max(0, cy - side // 2); tx0 = max(0, cx - side // 2)
    ty1 = min(H, ty0 + side);     tx1 = min(W, tx0 + side)
    ty0 = max(0, ty1 - side);     tx0 = max(0, tx1 - side)
    m224 = cv2.resize(mask_d[ty0:ty1, tx0:tx1], (Sz, Sz),
                      interpolation=cv2.INTER_NEAREST).astype(np.uint8)
    out["roi"]  = (cv2.resize(rgb_u8[ty0:ty1, tx0:tx1], (Sz, Sz), interpolation=cv2.INTER_AREA), m224)
    out["lung"] = (cv2.resize(masked[ty0:ty1, tx0:tx1], (Sz, Sz), interpolation=cv2.INTER_AREA), m224)
    return out

print("On-isleme hatti hazir.")

In [ ]:
# ── Oklüzyon maskeleri ───────────────────────────────────────────────────
CORNER_F = 0.22      # kose kare kenari (S orani)
BORDER_F = 0.08      # cevre serit genisligi
SUBDIA_F = 0.28      # alt bant yuksekligi
UPPER_F  = 0.20      # ust bant yuksekligi
CENTER_F = 0.20      # merkez sutun genisligi

def geom_masks(Sz):
    '''Goruntuden bagimsiz geometrik bolgeler -> {ad: bool maske}'''
    m = {}
    c = int(round(Sz * CORNER_F))
    a = np.zeros((Sz, Sz), bool)
    a[:c, :c] = a[:c, -c:] = a[-c:, :c] = a[-c:, -c:] = True
    m["corners"] = a
    b = int(round(Sz * BORDER_F))
    a = np.zeros((Sz, Sz), bool)
    a[:b, :] = a[-b:, :] = a[:, :b] = a[:, -b:] = True
    m["border"] = a
    a = np.zeros((Sz, Sz), bool); a[int(round(Sz * (1 - SUBDIA_F))):, :] = True
    m["subdiaphragm"] = a
    a = np.zeros((Sz, Sz), bool); a[:int(round(Sz * UPPER_F)), :] = True
    m["upper"] = a
    a = np.zeros((Sz, Sz), bool)
    lo = int(round(Sz * (0.5 - CENTER_F / 2))); hi = int(round(Sz * (0.5 + CENTER_F / 2)))
    a[:, lo:hi] = True
    m["center"] = a
    return m

GEOM = geom_masks(S)
print("Geometrik bolgelerin kapladigi alan:")
for k, v in GEOM.items():
    print(f"  {k:<14}: %{100*v.mean():.1f}")

def random_rect_mask(Sz, area_frac, rng):
    '''Verilen alan oranini kaplayan rastgele konumlu dikdortgen.'''
    area_frac = float(np.clip(area_frac, 0.01, 0.95))
    target = area_frac * Sz * Sz
    ar = np.exp(rng.uniform(np.log(0.5), np.log(2.0)))      # en/boy 0.5-2.0
    h = int(round(np.sqrt(target / ar))); w = int(round(target / max(h, 1)))
    h = int(np.clip(h, 1, Sz)); w = int(np.clip(w, 1, Sz))
    y = rng.integers(0, Sz - h + 1); x = rng.integers(0, Sz - w + 1)
    a = np.zeros((Sz, Sz), bool); a[y:y + h, x:x + w] = True
    return a

COND = ["baseline", "lungs", "non_lungs", "corners", "border", "subdiaphragm",
        "upper", "center", "rand_lungs", "rand_20", "rand_30"]
# Her kosulun alan-eslesmeli kontrolu (yorumda kullanilir)
CONTROL_OF = {"lungs": "rand_lungs", "non_lungs": None,
              "corners": "rand_20", "center": "rand_20", "upper": "rand_20",
              "border": "rand_30", "subdiaphragm": "rand_30"}

def build_variants(img224, mask224, rng):
    '''Bir kolun girdisinden 11 oklüzyon varyanti uretir (uint8 RGB listesi).'''
    out = []
    lung = mask224.astype(bool) if mask224 is not None else np.zeros((S, S), bool)
    masks = {"baseline": np.zeros((S, S), bool),
             "lungs": lung, "non_lungs": ~lung,
             **GEOM,
             "rand_lungs": random_rect_mask(S, max(lung.mean(), 0.02), rng),
             "rand_20": random_rect_mask(S, 0.20, rng),
             "rand_30": random_rect_mask(S, 0.30, rng)}
    for c in COND:
        v = img224.copy()
        v[masks[c]] = FILL
        out.append(v)
    return out, {c: float(masks[c].mean()) for c in COND}

def to_batch(vars_u8):
    x = torch.from_numpy(np.stack(vars_u8)).permute(0, 3, 1, 2).float() / 255.0
    return ((x - MEAN_T) / STD_T).to(device)

print(f"\n{len(COND)} kosul: {', '.join(COND)}")

In [ ]:
# ── Onizleme: oklüzyon bolgeleri ─────────────────────────────────────────
_p = None
for r, _, fs in os.walk(os.path.join(BASE, "test", "PNEUMONIA")):
    for f in sorted(fs):
        if f.lower().endswith((".jpeg", ".jpg", ".png")):
            _p = os.path.join(r, f); break
    if _p: break
rgb, gray = load_image_any(_p, CFG["orig_short_max"])
lung = lung_mask_ianpan(gray, gray.shape)
arms = prep_arms(rgb, lung, CFG)
rng0 = np.random.default_rng(SEED)
vars_u8, areas = build_variants(arms["raw"][0], arms["raw"][1], rng0)

fig, axes = plt.subplots(2, 6, figsize=(17, 6))
for ax, c, v in zip(axes.ravel(), COND, vars_u8):
    ax.imshow(v); ax.set_title(f"{c}\n%{100*areas[c]:.0f} alan", fontsize=8.5); ax.axis("off")
axes.ravel()[-1].axis("off")
fig.suptitle("Oklüzyon kosullari — A kolu girdisi uzerinde", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(WORK, "occ_fig_00_conditions.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Degerlendirme kumeleri ───────────────────────────────────────────────
def internal_test_items():
    '''Ablasyondaki test bolmesini birebir yeniden uret.'''
    rows = []
    for split in ["train", "val", "test"]:
        for cls in ["NORMAL", "PNEUMONIA"]:
            d = os.path.join(BASE, split, cls)
            if not os.path.isdir(d):
                continue
            for f in sorted(os.listdir(d)):
                if f.lower().endswith((".jpeg", ".jpg", ".png")):
                    rows.append({"split": split, "sinif": cls, "ad": f,
                                 "yol": os.path.join(d, f)})
    inv = pd.DataFrame(rows)
    sig = lambda p: f"{os.path.basename(p)}_{os.path.getsize(p)}"
    random.seed(SEED)
    tr = inv[inv.split == "train"]
    pool = inv[inv.split.isin(["val", "test"])]
    tr_sigs = set(sig(p) for p in tr["yol"])
    pool = pool[~pool["yol"].map(lambda p: sig(p) in tr_sigs)]
    npool = pool[pool.sinif == "NORMAL"].to_dict("records")
    ppool = pool[pool.sinif == "PNEUMONIA"].to_dict("records")
    random.shuffle(npool); random.shuffle(ppool)
    per = min(len(npool), len(ppool)) // 2
    val_s = npool[:per] + ppool[:per]
    test_s = npool[per:2 * per] + ppool[per:2 * per]
    random.shuffle(val_s); random.shuffle(test_s)
    fp = hashlib.md5("|".join(sorted(r["ad"] for r in test_s)).encode()).hexdigest()[:16]
    print(f"  ic test imzasi: {fp}  (beklenen 3a25cbb40ab471d7)")
    return [(r["yol"], 1 if r["sinif"] == "PNEUMONIA" else 0) for r in test_s]

def build_rsna_items(base, k):
    info = pd.read_csv(os.path.join(base, "stage_2_detailed_class_info.csv")).drop_duplicates("patientId")
    img_dir = os.path.join(base, "stage_2_train_images")
    pos = info[info["class"] == "Lung Opacity"]["patientId"].tolist()
    neg = info[info["class"] == "Normal"]["patientId"].tolist()
    rng = random.Random(SEED); rng.shuffle(pos); rng.shuffle(neg)
    kk = min(k, len(pos), len(neg))
    return ([(os.path.join(img_dir, p + ".dcm"), 1) for p in pos[:kk]] +
            [(os.path.join(img_dir, p + ".dcm"), 0) for p in neg[:kk]])

def build_nih_items(base, k):
    df = pd.read_csv(os.path.join(base, "Data_Entry_2017.csv"))
    lab = df["Finding Labels"].astype(str)
    ip = lab.apply(lambda s: "Pneumonia" in s.split("|"))
    inn = lab.apply(lambda s: s.strip() == "No Finding")
    index = {}
    for r, _, files in os.walk(base):
        for f in files:
            if f.lower().endswith(".png"):
                index[f] = os.path.join(r, f)
    pf = lambda m: [index[n] for n in df[m]["Image Index"].tolist() if n in index]
    pos, neg = pf(ip), pf(inn)
    rng = random.Random(SEED); rng.shuffle(pos); rng.shuffle(neg)
    kk = min(k, len(pos), len(neg))
    return [(p, 1) for p in pos[:kk]] + [(p, 0) for p in neg[:kk]]

DATASETS = [("Ic test", internal_test_items())]
if RSNA_BASE: DATASETS.append(("RSNA", build_rsna_items(RSNA_BASE, MAX_PER_CLASS)))
if NIH_BASE:  DATASETS.append(("NIH",  build_nih_items(NIH_BASE, MAX_PER_CLASS)))
for n, it in DATASETS:
    print(f"{n:<8}: {len(it)} goruntu")

In [ ]:
# ── Ana dongu: goruntu basina 1 segmentasyon, 11 kosul x 3 kol ───────────
@torch.inference_mode()
def run_occlusion(name, items):
    rng = np.random.default_rng(SEED)
    y, P = [], {(a, c): [] for a in ARMS for c in COND}
    AREA = {(a, c): [] for a in ARMS for c in COND}
    t0 = time.time()
    for i, (path, label) in enumerate(items):
        try:
            rgb, gray = load_image_any(path, CFG["orig_short_max"])
            lung = lung_mask_ianpan(gray, gray.shape)
            arms = prep_arms(rgb, lung, CFG)
        except Exception:
            continue
        y.append(label)
        for a in ARMS:
            img224, m224 = arms[a]
            vars_u8, areas = build_variants(img224, m224, rng)
            out = MODELS[a](to_batch(vars_u8))
            pr = torch.softmax(out, 1)[:, PNEU_IDX].cpu().numpy()
            for j, c in enumerate(COND):
                P[(a, c)].append(float(pr[j]))
                AREA[(a, c)].append(areas[c])
        if (i + 1) % 200 == 0:
            print(f"    [{name}] {i+1}/{len(items)} ({time.time()-t0:.0f} sn)")
    print(f"  [{name}] bitti: {len(y)} goruntu, {time.time()-t0:.0f} sn")
    return np.array(y), {k: np.array(v) for k, v in P.items()}, \
           {k: float(np.mean(v)) for k, v in AREA.items()}

RES = {}
for name, items in DATASETS:
    print(f"\n>>> {name} ({len(items)} goruntu x {len(COND)} kosul x 3 kol)...")
    RES[name] = run_occlusion(name, items)

In [ ]:
# ── Istatistik: DeLong (eslestirilmis) + bootstrap ───────────────────────
def _mr(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1); i = j
    T2 = np.empty(N); T2[J] = T + 1
    return T2

def _fd(ps, m):
    n = ps.shape[1] - m; k = ps.shape[0]
    tx = np.empty([k, m]); ty = np.empty([k, n]); tz = np.empty([k, m + n])
    for r in range(k):
        tx[r] = _mr(ps[r, :m]); ty[r] = _mr(ps[r, m:]); tz[r] = _mr(ps[r])
    a = tz[:, :m].sum(1) / m / n - (m + 1.) / 2. / n
    return a, np.cov((tz[:, :m] - tx) / n) / m + np.cov(1. - (tz[:, m:] - ty) / m) / n

def delong(y, p1, p2):
    y = np.asarray(y).astype(int); o = np.argsort(-y, kind="mergesort"); m = int(y.sum())
    a, c = _fd(np.vstack((p1, p2))[:, o], m); l = np.array([[1., -1.]])
    v = float(l.dot(c).dot(l.T))
    if v <= 0:
        return a[0], a[1], 0.0, 1.0
    z = float((a[0] - a[1]) / np.sqrt(v))
    return a[0], a[1], z, float(2 * (1 - sp_stats.norm.cdf(abs(z))))

def boot_delta(y, p_base, p_occ, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    y = np.asarray(y); ip, ineg = np.where(y == 1)[0], np.where(y == 0)[0]
    out = np.empty(n_boot)
    for b in range(n_boot):
        ii = np.concatenate([rng.choice(ip, len(ip), True), rng.choice(ineg, len(ineg), True)])
        f1, t1, _ = roc_curve(y[ii], p_base[ii]); f2, t2, _ = roc_curve(y[ii], p_occ[ii])
        out[b] = auc(f1, t1) - auc(f2, t2)
    return out

rows = []
for name, (y, P, AREA) in RES.items():
    for a in ARMS:
        fb, tb, _ = roc_curve(y, P[(a, "baseline")]); A0 = auc(fb, tb)
        for c in COND:
            if c == "baseline":
                continue
            f1, t1, _ = roc_curve(y, P[(a, c)]); A1 = auc(f1, t1)
            _, _, z, p = delong(y, P[(a, "baseline")], P[(a, c)])
            bd = boot_delta(y, P[(a, "baseline")], P[(a, c)])
            rows.append({"kume": name, "kol": a, "kosul": c,
                         "alan": AREA[(a, c)], "AUC": A1, "dAUC": A0 - A1,
                         "GA alt": float(np.percentile(bd, 2.5)),
                         "GA ust": float(np.percentile(bd, 97.5)),
                         "DeLong p": p})
df = pd.DataFrame(rows)

# net etki = dAUC(bolge) - dAUC(alan eslesmeli rastgele)
def net_effect(r):
    ctrl = CONTROL_OF.get(r["kosul"])
    if not ctrl:
        return np.nan
    m = df[(df.kume == r["kume"]) & (df.kol == r["kol"]) & (df.kosul == ctrl)]
    return r["dAUC"] - float(m["dAUC"].iloc[0]) if len(m) else np.nan
df["net etki"] = df.apply(net_effect, axis=1)

pd.set_option("display.float_format", lambda v: f"{v:.4f}")
pd.set_option("display.width", 220)
print("=" * 120)
print("  OKLUZYON SONUCLARI  (dAUC = baseline - kapali;  net etki = dAUC - alan eslesmeli rastgele)")
print("=" * 120)
for name in RES:
    print(f"\n--- {name} ---")
    print(df[df.kume == name].drop(columns=["kume"]).to_string(index=False))

In [ ]:
# ── Tutarlilik denetimleri ───────────────────────────────────────────────
print("=" * 84)
print("  TUTARLILIK DENETIMLERI")
print("=" * 84)
ok = True
for name in RES:
    r1 = df[(df.kume == name) & (df.kol == "lung") & (df.kosul == "non_lungs")]["dAUC"]
    r2 = df[(df.kume == name) & (df.kol == "lung") & (df.kosul == "lungs")]["dAUC"]
    a = float(r1.iloc[0]); b = float(r2.iloc[0])
    c1 = abs(a) < 0.01; c2 = b > 0.15
    ok &= (c1 and c2)
    print(f"  {name:<8} C/non_lungs dAUC={a:+.4f} {'OK (etkisiz beklenir)' if c1 else 'BEKLENMEDIK'}"
          f"   |   C/lungs dAUC={b:+.4f} {'OK (yikici beklenir)' if c2 else 'BEKLENMEDIK'}")
print("=" * 84)
print("  Ikisi de gecerse hat dogru kurulmustur." if ok else
      "  UYARI: beklenen davranis saglanmadi - sonuclari yorumlamadan once inceleyin.")

# ── Ana bulgu ozeti ──────────────────────────────────────────────────────
print("\n" + "=" * 96)
print("  ANA SORU: akcigeri kapatmak, ayni alanda rastgele bir bolgeyi kapatmaktan fazla zarar veriyor mu?")
print("=" * 96)
print(f"  {'kume':<10}{'kol':<7}{'dAUC(lungs)':>13}{'dAUC(rand)':>13}{'net etki':>11}{'DeLong p':>11}")
for name in RES:
    for a in ARMS:
        L = df[(df.kume == name) & (df.kol == a) & (df.kosul == "lungs")].iloc[0]
        R = df[(df.kume == name) & (df.kol == a) & (df.kosul == "rand_lungs")].iloc[0]
        print(f"  {name:<10}{a:<7}{L['dAUC']:>13.4f}{R['dAUC']:>13.4f}"
              f"{L['net etki']:>11.4f}{L['DeLong p']:>11.4f}")
print("=" * 96)

In [ ]:
# ── Figur: net etki, kosul x kol ─────────────────────────────────────────
SHOW = ["lungs", "non_lungs", "corners", "border", "subdiaphragm", "upper", "center"]
fig, axes = plt.subplots(1, len(RES), figsize=(5.4 * len(RES), 5.0), squeeze=False)
for ax, name in zip(axes[0], RES):
    xs = np.arange(len(SHOW)); w = 0.26
    for k, a in enumerate(ARMS):
        vals = []
        for c in SHOW:
            r = df[(df.kume == name) & (df.kol == a) & (df.kosul == c)]
            v = float(r["net etki"].iloc[0]) if not np.isnan(r["net etki"].iloc[0]) \
                else float(r["dAUC"].iloc[0])
            vals.append(v)
        ax.barh(xs + (k - 1) * w, vals, height=w * 0.86, color=ARM_COLOR[a],
                edgecolor="white", linewidth=1.2, label=ARM_LABEL[a], zorder=3)
    ax.axvline(0, color="#5A686F", lw=1.1, zorder=4)
    ax.set_yticks(xs); ax.set_yticklabels(SHOW, fontsize=9)
    ax.invert_yaxis(); ax.set_xlabel("net etki  (dAUC - alan eslesmeli rastgele)")
    ax.set_title(name); ax.xaxis.grid(True, alpha=.5); ax.set_axisbelow(True)
axes[0][0].legend(loc="lower right", fontsize=8)
fig.suptitle("Bolge kapatmanin nedensel etkisi — sifira yakin = o bolge karara katkida bulunmuyor",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(WORK, "occ_fig_01_net_effect.png"), dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# ── Kayit ────────────────────────────────────────────────────────────────
df.to_csv(os.path.join(WORK, "occlusion_results.csv"), index=False)
np.savez_compressed(os.path.join(WORK, "occlusion_predictions.npz"),
                    **{f"{n}__y": RES[n][0] for n in RES},
                    **{f"{n}__{a}__{c}": RES[n][1][(a, c)]
                       for n in RES for a in ARMS for c in COND})
with open(os.path.join(WORK, "occlusion_summary.json"), "w", encoding="utf-8") as f:
    json.dump({"kosullar": COND, "kontrol_eslesmesi": CONTROL_OF,
               "alan_oranlari": {k: float(v.mean()) for k, v in GEOM.items()},
               "dolgu_degeri": FILL,
               "n": {n: int(len(RES[n][0])) for n in RES}}, f, indent=2, ensure_ascii=False)
print("Kaydedilenler:")
for f in sorted(os.listdir(WORK)):
    if f.startswith("occ"):
        print(f"  {f:<34} {os.path.getsize(os.path.join(WORK, f))/1e6:>7.2f} MB")

## Sonucun okunması

Yorumun tamamı **net etki** sütunundadır; ham `dAUC` tek başına yanıltıcıdır çünkü
herhangi bir bölgeyi kapatmak girdi dağılımını bozar ve bir miktar düşüş yaratır.

**A kolu (segmentasyonsuz) için beklenen bulgu:**

- `lungs` net etkisi ≈ 0 → akciğeri silmek, aynı alanda rastgele bir dikdörtgeni
  silmekten fazla zarar vermiyor. Dikkat analizinin (LFR 0,011) nedensel doğrulaması.
- `corners`, `border`, `subdiaphragm` net etkisi > 0 → karar bu bölgelere dayanıyor.

Bu iki sonuç birlikte, **korelasyonel bir gözlemi nedensel bir iddiaya** dönüştürür:
*"Model akciğere bakmıyor"* yerine *"Akciğeri tamamen silmek modelin başarımını
düşürmüyor; buna karşılık çevresel bölgeleri silmek düşürüyor."*

**C kolu için beklenen tersidir:** `lungs` yıkıcı, `non_lungs` etkisiz. Tutarlılık
denetimi bunu otomatik sınar.

**Net etki her yerde ≈ 0 çıkarsa** model bilgiyi geniş biçimde dağıtmış demektir;
bu durumda oklüzyon testi ayrım üretmez ve makalede yalnızca dikkat temelli kanıt
(LFR) kullanılmalıdır — testin negatif sonucu da dürüstçe raporlanır.

### Kısıtlar

1. **Oklüzyon dağıtım dışı girdi üretir.** Gri dolgu, modelin eğitimde gördüğü bir
   örüntü değildir (C kolu hariç). Alan eşleşmeli kontroller bu etkiyi büyük ölçüde
   giderir ancak tamamen ortadan kaldırmaz.
2. **Bölgeler geometriktir**, anatomik değil (`subdiaphragm` bandı hastanın gerçek
   diyafram düzeyiyle birebir örtüşmeyebilir). Akciğer maskesi ise anatomiktir.
3. **Tek eğitim tohumu** kullanılmıştır; etki büyüklükleri tohuma göre oynayabilir,
   ancak beklenen örüntü (net etki ≈ 0 ↔ > 0) niteliksel bir ayrımdır.